# Pipeline Demo and Evidence Notebook

This executed notebook collects intermediate evidence for the Lab 04 streaming pipeline. It reads the
generated manifest, Kafka samples, verification logs, and screenshot artifacts used by the Jupyter
Book report.

The notebook is not a replacement for the two-terminal streaming demo. Instead, it summarizes the
recorded run so the submitted book contains executable cells with real intermediate outputs:
repository discovery counts, parser output, Kafka sample payloads, MongoDB/Neo4j recorded
verification results, replay before/after results, checkpoint evidence, and artifact availability.

In [1]:
from pathlib import Path
import json
import re
from pprint import pprint

try:
    import pandas as pd
except ImportError:
    pd = None

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
elif PROJECT_ROOT.name == "book":
    PROJECT_ROOT = PROJECT_ROOT.parent

BOOK_DIR = PROJECT_ROOT / "book"
LOG_DIR = BOOK_DIR / "logs"
KAFKA_DIR = BOOK_DIR / "kafka"
IMAGE_DIR = BOOK_DIR / "images"
MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "discovered_files.json"

{
    "project_root": str(PROJECT_ROOT),
    "book_dir": str(BOOK_DIR),
    "manifest_exists": MANIFEST_PATH.exists(),
    "log_dir_exists": LOG_DIR.exists(),
    "kafka_dir_exists": KAFKA_DIR.exists(),
    "image_dir_exists": IMAGE_DIR.exists(),
}

{'project_root': '/home/macht/Dev/HCMUS/Streaming',
 'book_dir': '/home/macht/Dev/HCMUS/Streaming/book',
 'manifest_exists': True,
 'log_dir_exists': True,
 'kafka_dir_exists': True,
 'image_dir_exists': True}

## Evidence artifact availability

In [2]:
required_artifacts = [
    "book/logs/check_infra.log",
    "book/logs/create_topics.log",
    "book/logs/terminal_1_spark_latest.log",
    "book/logs/terminal_2_pipeline_latest.log",
    "book/logs/identity_replay_verification.log",
    "book/logs/checkpoint_resume.log",
    "book/logs/mongodb_indexes.log",
    "book/logs/pytest.log",
    "book/kafka/nodes_sample.json",
    "book/kafka/edges_sample.json",
    "book/kafka/metadata_sample.json",
    "book/kafka/errors_sample.json",
    "book/images/architecture.svg",
    "book/images/neo4j-counts.png",
    "book/images/neo4j-graph-view.png",
    "book/images/mongo-source-metadata-list.png",
    "book/images/spark-structured-streaming-ui.png",
    "book/images/node_edge_count_lab_replay_probe.png",
]

artifact_rows = []
for rel in required_artifacts:
    path = PROJECT_ROOT / rel
    artifact_rows.append({
        "artifact": rel,
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else None,
    })

if pd is not None:
    display(pd.DataFrame(artifact_rows))
else:
    pprint(artifact_rows)

,artifact,exists,size_bytes
0,book/logs/check_infra.log,True,1162
1,book/logs/create_topics.log,True,931
2,book/logs/terminal_1_spark_latest.log,True,45311
3,book/logs/terminal_2_pipeline_latest.log,True,44384
4,book/logs/identity_replay_verification.log,True,839
5,book/logs/checkpoint_resume.log,True,262
6,book/logs/mongodb_indexes.log,True,492
7,book/logs/pytest.log,True,90
8,book/kafka/nodes_sample.json,True,456
9,book/kafka/edges_sample.json,True,493


## Repository discovery manifest

In [3]:
manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else {}
paths = manifest.get("files") or manifest.get("paths") or manifest.get("python_files") or []
total_files = manifest.get("total_files", len(paths))

summary = {
    "manifest_path": str(MANIFEST_PATH.relative_to(PROJECT_ROOT)),
    "repo_path": manifest.get("repo_path"),
    "total_files": total_files,
    "sample_count_displayed": min(10, len(paths)),
}
summary

{'manifest_path': 'data/processed/discovered_files.json',
 'repo_path': 'data/raw/accelerate',
 'total_files': 99,
 'sample_count_displayed': 10}

In [4]:
sample_paths = paths[:10]
if pd is not None:
    display(pd.DataFrame({"sample_discovered_file": sample_paths}))
else:
    pprint(sample_paths)

,sample_discovered_file
0,benchmarks/big_model_inference/big_model_infer...
1,benchmarks/big_model_inference/measures_util.py
2,benchmarks/fp8/ms_amp/ddp.py
3,benchmarks/fp8/ms_amp/distrib_deepspeed.py
4,benchmarks/fp8/ms_amp/fp8_utils.py
5,benchmarks/fp8/ms_amp/non_distributed.py
6,benchmarks/fp8/torchao/ddp.py
7,benchmarks/fp8/torchao/distrib_deepspeed.py
8,benchmarks/fp8/torchao/fp8_utils.py
9,benchmarks/fp8/torchao/fsdp.py


## Parser output summary

In [5]:
terminal2_path = LOG_DIR / "terminal_2_pipeline_latest.log"
terminal2_log = terminal2_path.read_text(errors="replace") if terminal2_path.exists() else ""
plain_terminal2 = re.sub(r"\x1b\[[0-9;]*m", "", terminal2_log)
collapsed_terminal2 = re.sub(r"\s+", " ", plain_terminal2)

discovered_match = re.search(r"Discovered\s+(\d+)\s+Python files", collapsed_terminal2)
finished_matches = re.findall(
    r"Finished:\s+successful=(\d+)\s+failed=(\d+)", collapsed_terminal2
)

parsed_rows = []
for match in re.finditer(
    r"Parsed\s+(.+?):\s+nodes=(\d+)\s+edges=(\d+)\s+metadata=(\d+)",
    collapsed_terminal2,
):
    parsed_rows.append({
        "file_path": match.group(1).strip(),
        "nodes": int(match.group(2)),
        "edges": int(match.group(3)),
        "metadata": int(match.group(4)),
    })

successful, failed = finished_matches[-1] if finished_matches else (None, None)
parser_summary = {
    "log_exists": terminal2_path.exists(),
    "discovered_files_in_log": int(discovered_match.group(1)) if discovered_match else None,
    "successful": int(successful) if successful is not None else None,
    "failed": int(failed) if failed is not None else None,
    "parsed_rows_extracted": len(parsed_rows),
}
parser_summary

{'log_exists': True,
 'discovered_files_in_log': 99,
 'successful': 99,
 'failed': 0,
 'parsed_rows_extracted': 100}

In [6]:
top_rows = sorted(parsed_rows, key=lambda row: row["nodes"], reverse=True)[:10]
if pd is not None:
    display(pd.DataFrame(top_rows))
else:
    pprint(top_rows)

,file_path,nodes,edges,metadata
0,src/accelerate/accelerator.py,15837,32872,1
1,src/accelerate/utils/dataclasses.py,13574,27527,1
2,src/accelerate/utils/modeling.py,11772,24763,1
3,src/accelerate/utils/megatron_lm.py,6319,13208,1
4,src/accelerate/data_loader.py,5965,12286,1
5,src/accelerate/utils/launch.py,5272,10912,1
6,src/accelerate/commands/launch.py,5041,10658,1
7,src/accelerate/state.py,4968,10070,1
8,src/accelerate/utils/fsdp_utils.py,4485,9364,1
9,src/accelerate/tracking.py,4238,8717,1


## Kafka message samples

In [7]:
def load_json_event(path: Path):
    raw = path.read_text(errors="replace").strip()
    candidate = raw.split("\t", 1)[1].strip() if "\t" in raw else raw
    return json.loads(candidate)


sample_files = [
    ("cpg.nodes.v1", KAFKA_DIR / "nodes_sample.json"),
    ("cpg.edges.v1", KAFKA_DIR / "edges_sample.json"),
    ("cpg.metadata.v1", KAFKA_DIR / "metadata_sample.json"),
    ("cpg.errors.v1", KAFKA_DIR / "errors_sample.json"),
]

kafka_rows = []
events = {}
for topic, path in sample_files:
    if path.exists():
        try:
            event = load_json_event(path)
            events[topic] = event
            kafka_rows.append({
                "topic": topic,
                "exists": True,
                "schema_version": event.get("schema_version"),
                "repo_name": event.get("repo_name"),
                "file_path": event.get("file_path"),
                "identity": event.get("node_id") or event.get("edge_id") or event.get("metadata_id") or event.get("error_type"),
                "event_type": event.get("node_type") or event.get("edge_type") or event.get("status") or event.get("error_type"),
                "status": event.get("status", "ok"),
            })
        except (json.JSONDecodeError, IndexError) as exc:
            kafka_rows.append({"topic": topic, "exists": True, "parse_error": str(exc)})
    else:
        kafka_rows.append({"topic": topic, "exists": False})

if pd is not None:
    display(pd.DataFrame(kafka_rows))
else:
    pprint(kafka_rows)

,topic,exists,schema_version,repo_name,file_path,identity,event_type,status
0,cpg.nodes.v1,True,1.0,accelerate,benchmarks/big_model_inference/big_model_infer...,38e41e11cb41c8fe7ec4ceb45a3aaa640e5639df205125...,alias,ok
1,cpg.edges.v1,True,1.0,accelerate,benchmarks/big_model_inference/big_model_infer...,e5077591dc0c165d2a86d3476884c1127a3aed08154d38...,AST,ok
2,cpg.metadata.v1,True,1.0,accelerate,benchmarks/big_model_inference/big_model_infer...,193be55d48d094dd2dabd8f933ffc82c76bef578804d6a...,parsed,parsed
3,cpg.errors.v1,True,1.0,accelerate,src/accelerate/_lab_parser_error.py,SyntaxError,failed,failed


In [8]:
metadata_event = events.get("cpg.metadata.v1", {})
metadata_focus = {
    key: metadata_event.get(key)
    for key in [
        "repo_name", "file_path", "line_count", "function_count",
        "class_count", "import_count", "node_count", "edge_count", "status",
    ]
}
metadata_focus

{'repo_name': 'accelerate',
 'file_path': 'benchmarks/big_model_inference/big_model_inference.py',
 'line_count': 143,
 'function_count': 2,
 'class_count': 0,
 'import_count': 7,
 'node_count': 877,
 'edge_count': 1849,
 'status': 'parsed'}

## Infrastructure and connector status

In [9]:
infra_path = LOG_DIR / "check_infra.log"
check_infra = infra_path.read_text(errors="replace") if infra_path.exists() else ""

infra_summary = {
    "log_exists": infra_path.exists(),
    "has_kafka": "cpg-kafka" in check_infra,
    "has_neo4j": "cpg-neo4j" in check_infra,
    "has_mongodb": "cpg-mongodb" in check_infra,
    "has_kafka_connect": "cpg-kafka-connect" in check_infra,
    "nodes_topic": "cpg.nodes.v1" in check_infra,
    "edges_topic": "cpg.edges.v1" in check_infra,
    "metadata_topic": "cpg.metadata.v1" in check_infra,
    "errors_topic": "cpg.errors.v1" in check_infra,
    "neo4j_connector_running": '"state":"RUNNING"' in check_infra or '"state": "RUNNING"' in check_infra,
}
infra_summary

{'log_exists': True,
 'has_kafka': True,
 'has_neo4j': True,
 'has_mongodb': True,
 'has_kafka_connect': True,
 'nodes_topic': True,
 'edges_topic': True,
 'metadata_topic': True,
 'errors_topic': True,
 'neo4j_connector_running': True}

## MongoDB metadata identity indexes

In [10]:
indexes_path = LOG_DIR / "mongodb_indexes.log"
indexes = []
index_parse_error = None
if indexes_path.exists():
    try:
        indexes = json.loads(indexes_path.read_text(errors="replace"))
    except json.JSONDecodeError as exc:
        index_parse_error = str(exc)

index_rows = [
    {"name": idx.get("name"), "key": idx.get("key"), "unique": idx.get("unique", False)}
    for idx in indexes
]
if not indexes_path.exists() or index_parse_error:
    index_rows.append({
        "name": None,
        "key": None,
        "unique": False,
        "exists": indexes_path.exists(),
        "parse_error": index_parse_error,
    })

if pd is not None:
    display(pd.DataFrame(index_rows))
else:
    pprint(index_rows)

,name,key,unique
0,_id_,{'_id': 1},False
1,metadata_id_1,{'metadata_id': 1},True
2,file_hash_1,{'file_hash': 1},False
3,event_time_-1,{'event_time': -1},False
4,repo_name_1_file_path_1,"{'repo_name': 1, 'file_path': 1}",False


## Recorded database verification results

In [11]:
def last_int(pattern):
    matches = re.findall(pattern, collapsed_terminal2, flags=re.IGNORECASE)
    return int(matches[-1]) if matches else None


database_summary = {
    "mongodb_metadata_documents": last_int(r"MongoDB metadata documents:\s*(\d+)"),
    "mongodb_duplicate_metadata_id_groups": last_int(r"Duplicate metadata_id groups:\s*(\d+)"),
    "mongodb_duplicate_repo_file_groups": last_int(r"Duplicate repo/file groups:\s*(\d+)"),
    "neo4j_total_nodes": last_int(r"Neo4j totals:\s*nodes=(\d+)"),
    "neo4j_total_edges": last_int(r"Neo4j totals:\s*nodes=\d+\s+edges=(\d+)"),
    "neo4j_duplicate_node_ids": last_int(r"Duplicate node IDs:\s*(\d+)"),
    "neo4j_duplicate_edge_ids": last_int(r"Duplicate edge IDs:\s*(\d+)"),
    "neo4j_unresolved_placeholders": last_int(r"Unresolved placeholder nodes:\s*(\d+)"),
}
if pd is not None:
    display(pd.DataFrame([{"metric": key, "value": value} for key, value in database_summary.items()]))
else:
    pprint(database_summary)

,metric,value
0,mongodb_metadata_documents,99
1,mongodb_duplicate_metadata_id_groups,0
2,mongodb_duplicate_repo_file_groups,0
3,neo4j_total_nodes,263154
4,neo4j_total_edges,626917
5,neo4j_duplicate_node_ids,0
6,neo4j_duplicate_edge_ids,0
7,neo4j_unresolved_placeholders,0


## Modified-file replay verification

In [12]:
replay_path = LOG_DIR / "identity_replay_verification.log"
replay_log = replay_path.read_text(errors="replace") if replay_path.exists() else ""
replay_values = {}
for line in replay_log.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        replay_values[key.strip()] = value.strip()

important_replay_keys = [
    "target_file", "file_hash_changed",
    "mongodb_document_count_before", "mongodb_document_count_after",
    "mongodb_document_count_delta", "mongodb_file_hash_matches_replay",
    "neo4j_target_nodes_before", "neo4j_target_nodes_after",
    "neo4j_target_edges_before", "neo4j_target_edges_after",
    "duplicate_node_id_groups", "duplicate_edge_id_groups",
    "duplicate_metadata_id_groups", "duplicate_repo_file_groups",
]
replay_summary = [{"metric": key, "value": replay_values.get(key)} for key in important_replay_keys]
if pd is not None:
    display(pd.DataFrame(replay_summary))
else:
    pprint(replay_summary)

,metric,value
0,target_file,src/accelerate/_lab_replay_probe.py
1,file_hash_changed,True
2,mongodb_document_count_before,99
3,mongodb_document_count_after,99
4,mongodb_document_count_delta,+0
5,mongodb_file_hash_matches_replay,True
6,neo4j_target_nodes_before,14
7,neo4j_target_nodes_after,14
8,neo4j_target_edges_before,27
9,neo4j_target_edges_after,26


In [13]:
before_after = [
    {
        "metric": "MongoDB documents",
        "before": replay_values.get("mongodb_document_count_before"),
        "after": replay_values.get("mongodb_document_count_after"),
        "expected": "unchanged",
    },
    {
        "metric": "Neo4j target nodes",
        "before": replay_values.get("neo4j_target_nodes_before"),
        "after": replay_values.get("neo4j_target_nodes_after"),
        "expected": "no duplicate node growth",
    },
    {
        "metric": "Neo4j target edges",
        "before": replay_values.get("neo4j_target_edges_before"),
        "after": replay_values.get("neo4j_target_edges_after"),
        "expected": "updated graph structure",
    },
]
if pd is not None:
    display(pd.DataFrame(before_after))
else:
    pprint(before_after)

,metric,before,after,expected
0,MongoDB documents,99,99,unchanged
1,Neo4j target nodes,14,14,no duplicate node growth
2,Neo4j target edges,27,26,updated graph structure


## Spark checkpoint resume evidence

In [14]:
checkpoint_path = LOG_DIR / "checkpoint_resume.log"
checkpoint_log = checkpoint_path.read_text(errors="replace") if checkpoint_path.exists() else ""
checkpoint_values = {}
for line in checkpoint_log.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        checkpoint_values[key.strip()] = value.strip()

checkpoint_values if checkpoint_values else {
    "exists": checkpoint_path.exists(),
    "message": "Checkpoint evidence file is missing or empty",
}

{'checkpoint_location': 'outputs/checkpoints/mongodb_metadata',
 'checkpoint_exists': 'True',
 'checkpoint_artifacts_before': '21',
 'checkpoint_artifacts_after': '21',
 'metadata_count_before': '99',
 'metadata_count_after': '99',
 'result': 'PASSED checkpoint resumed without duplicating unchanged metadata'}

In [15]:
if pd is not None:
    display(pd.DataFrame([
        {"metric": key, "value": value}
        for key, value in checkpoint_values.items()
    ]))
else:
    pprint(checkpoint_values)

,metric,value
0,checkpoint_location,outputs/checkpoints/mongodb_metadata
1,checkpoint_exists,True
2,checkpoint_artifacts_before,21
3,checkpoint_artifacts_after,21
4,metadata_count_before,99
5,metadata_count_after,99
6,result,PASSED checkpoint resumed without duplicating ...


## Final evidence assessment

In [16]:
pytest_path = LOG_DIR / "pytest.log"
pytest_text = pytest_path.read_text(errors="replace") if pytest_path.exists() else ""
assessment = {
    "repository_discovery_ok": total_files == 99,
    "parser_baseline_ok": parser_summary.get("successful") == 99 and parser_summary.get("failed") == 0,
    "kafka_samples_available": all(path.exists() for _, path in sample_files),
    "connector_running_in_log": infra_summary["neo4j_connector_running"],
    "mongodb_no_duplicate_replay": replay_values.get("mongodb_document_count_delta") in {"+0", "0"},
    "neo4j_duplicate_nodes_zero": replay_values.get("duplicate_node_id_groups") == "0",
    "neo4j_duplicate_edges_zero": replay_values.get("duplicate_edge_id_groups") == "0",
    "checkpoint_passed": "PASSED" in checkpoint_values.get("result", ""),
    "pytest_core_passed": "19 passed" in pytest_text,
}
assessment

{'repository_discovery_ok': True,
 'parser_baseline_ok': True,
 'kafka_samples_available': True,
 'connector_running_in_log': True,
 'mongodb_no_duplicate_replay': True,
 'neo4j_duplicate_nodes_zero': True,
 'neo4j_duplicate_edges_zero': True,
 'checkpoint_passed': True,
 'pytest_core_passed': True}

## Reflection

**What worked:** The tracked artifacts show repository discovery, one-file-at-a-time parser counts,
all four Kafka event categories, connector status, MongoDB identity indexes, database verification,
modified-file replay, checkpoint resume behavior, and passing tests.

**Evidence boundary:** These cells summarize recorded verification output; they do not issue live
database queries or replace the two-terminal streaming demonstration.

**Remaining limitations:** CFG, DFG, and CALL construction is intentionally lightweight. Modified-file
Neo4j replay uses file-scoped cleanup before replacement events are republished, so it is a lab
verification protocol rather than a fully event-driven deletion lifecycle. The checkpoint evidence
shows stable metadata during the recorded resume/idle check, not exact Kafka offset proof.